In [ ]:
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
reference_points = np.load('/home/ray/projects/helicopter/notebooks/measured_points.npy')

In [ ]:
# reference_points = np.array([[    0.31536,    0.082801,   -0.038738],
#  [    0.32249,    0.026563,   -0.033961],
#  [     0.3138,    0.058204,    -0.02006],
#  [    0.31732,    0.063587,   -0.037751],
#  [    0.32169,    0.031875,   -0.022396],
#  [    0.32017,    0.097442,   -0.032178],
#  [     0.3147,     0.18443,   -0.064938],
#  [     0.3196,    0.081567,   -0.027579],
#  [    0.31745,     0.17794,   -0.045529]])

In [ ]:
reference_mean = reference_points.mean(axis=0)

In [ ]:
%matplotlib ipympl

plt.close('all')

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')
ax.scatter(reference_points[:, 0], reference_points[:, 1], reference_points[:, 2], color='k')
ax.set_xlim(reference_mean[0] - 0.15, reference_mean[0] + .15)
ax.set_ylim(reference_mean[1] - 0.15, reference_mean[1] + .15)
ax.set_zlim(reference_mean[2] - 0.15, reference_mean[2] + .15)

In [ ]:
import quaternion

In [ ]:
ref_dist = np.linalg.norm(reference_mean)
q = quaternion.from_rotation_vector([0, 0, np.pi / 4])

In [ ]:
modified_points = quaternion.rotate_vectors(q, reference_points)

modified_mean = modified_points.mean(axis=0)
t = reference_mean - modified_mean
modified_points = modified_points + t

In [ ]:
%matplotlib ipympl

plt.close('all')

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')
ax.scatter(reference_points[:, 0], reference_points[:, 1], reference_points[:, 2], color='k')
ax.set_xlim(reference_mean[0] - 0.2, reference_mean[0] + .2)
ax.set_ylim(reference_mean[1] - 0.2, reference_mean[1] + .2)
ax.set_zlim(reference_mean[2] - 0.2, reference_mean[2] + .2)

ax.scatter(modified_points[:, 0], modified_points[:, 1], modified_points[:, 2], color='r')

In [ ]:
def kabsch(p: np.ndarray, q: np.ndarray):
    """
    Computes the optimal rotation from measured to reference points, which should capture the camera frame to world frame transformation.
    Args:
        p: matched points in the world frame
        q: points in the camera frame

    Returns:

    """
    n = p.shape[0]
    m_c = np.sum(p, axis=0) / n
    r_c = np.sum(q, axis=0) / n

    measured_points_centered = p - m_c
    reference_points_centered = q - r_c

    covar = measured_points_centered.transpose() @ reference_points_centered
    U, s, Vh = np.linalg.svd(covar)

    rotation_matrix = Vh.T @ U.T

    if np.linalg.det(rotation_matrix) < 0:
        Vh_fixed = Vh.copy()
        Vh_fixed[2, :] *= -1
        rotation_matrix = Vh_fixed.T @ U.T

    translation = r_c - (rotation_matrix @ m_c)

    return rotation_matrix, translation

In [ ]:
R, t_pred = kabsch(modified_points, reference_points)
q_pred = quaternion.from_rotation_matrix(R)

In [ ]:
from helicopter.vision.measurement.camera_state_handler import CameraStateHandler
csh = CameraStateHandler()

In [ ]:
success, R_csh, t_csh = CameraStateHandler.ransac_visual_pose(modified_points, reference_points)

In [ ]:
q_csh = quaternion.from_rotation_matrix(R_csh)

In [ ]:
from helicopter.vision.point_detection.model_training import BlobPointDetector
from vision.point_detection.point_handler import PointHandler

ph = PointHandler(
        detector=BlobPointDetector(),
        queue_len=75)

In [ ]:
corrected_points = ph.correct_points(modified_points, t_csh, q_csh)

In [ ]:
%matplotlib ipympl

plt.close('all')

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')
ax.scatter(reference_points[:, 0], reference_points[:, 1], reference_points[:, 2], color='k')
ax.set_xlim(reference_mean[0] - 0.2, reference_mean[0] + .2)
ax.set_ylim(reference_mean[1] - 0.2, reference_mean[1] + .2)
ax.set_zlim(reference_mean[2] - 0.2, reference_mean[2] + .2)

ax.scatter(corrected_points[:, 0], corrected_points[:, 1], corrected_points[:, 2], color='g')